# 03. 품질 점검과 분석 대상 확정

**입력**: `dataSets/starfield_analysis.csv` (리뷰 + 업적 병합 테이블, 72,271행)
**출력**: `dataSets/reviews_for_embedding.csv` (steamid + 리뷰 원문 + voted_up)

이 노트북이 하는 일은 두 가지다.

1. 데이터가 어떻게 생겼는지 숫자로 확인한다 (PPT 데이터셋 소개 페이지의 수치).
2. 분석에 쓸 수 없는 행을 빼고, 뒤의 모든 노트북이 같은 행 집합을 쓰도록 id 목록을 저장한다.

In [1]:
import pandas as pd

df = pd.read_csv('dataSets/starfield_analysis.csv', low_memory=False)
n_before = len(df)
print('행', n_before, '/ 열', df.shape[1])

# Steam은 계정당 게임 하나에 리뷰 하나만 쓸 수 있다 → steamid가 리뷰의 고유키
print('steamid 중복 없음:', df['steamid'].is_unique)

행 72271 / 열 44
steamid 중복 없음: True


## 1. 데이터셋 소개 수치

비율은 "조건에 맞는 행 수 / 전체 행 수"로 계산한다.

In [2]:
# 리뷰 관련
print('긍정률(voted_up 평균):', round(df['voted_up'].mean() * 100, 1), '%')
print('언어 종류:', df['language'].nunique())
print('리뷰 텍스트 NaN:', df['review'].isna().sum(), '건')
print('리뷰 최장 길이:', int(df['review'].str.len().max()), '자')

ts = pd.to_datetime(df['ts_created'], unit='s')
print('리뷰 작성 기간:', ts.min().date(), '~', ts.max().date())

긍정률(voted_up 평균): 57.0 %
언어 종류: 29
리뷰 텍스트 NaN: 192 건
리뷰 최장 길이: 8000 자
리뷰 작성 기간: 2023-09-06 ~ 2026-09-07


In [3]:
# 리뷰에 대한 타인의 반응 (분석에는 안 씀 — 부록용)
print('votes_up 최대:', df['votes_up'].max(), '/ 0인 비율:', round((df['votes_up'] == 0).mean() * 100, 1), '%')
print('votes_funny 최대:', df['votes_funny'].max(), '/ 0인 비율:', round((df['votes_funny'] == 0).mean() * 100, 1), '%')
print('comment_count 최대:', df['comment_count'].max(), '/ 0인 비율:', round((df['comment_count'] == 0).mean() * 100, 1), '%')

dev = df['timestamp_dev_responded'].dropna()
dev_date = pd.to_datetime(dev, unit='s')
print('개발사 답변:', len(dev), '건 (', round(len(dev) / len(df) * 100, 1), '% ),', dev_date.min().date(), '~', dev_date.max().date())

votes_up 최대: 15871 / 0인 비율: 46.6 %
votes_funny 최대: 3456 / 0인 비율: 82.5 %
comment_count 최대: 625 / 0인 비율: 96.5 %
개발사 답변: 1355 건 ( 1.9 % ), 2023-09-19 ~ 2023-11-28


In [4]:
# 프로필·플레이타임
print('games_owned 중앙값:', df['games_owned'].median(), '/ 최대:', df['games_owned'].max())
print('num_reviews 최소:', df['num_reviews'].min(), '/ 중앙값:', df['num_reviews'].median(), '/ 최대:', df['num_reviews'].max())

at_review_h = df['playtime_at_review_min'].median() / 60
forever_h = df['playtime_forever_min'].median() / 60
print('리뷰 시점 플레이타임 중앙값:', df['playtime_at_review_min'].median(), '분 (약', round(at_review_h), 'h ) / 최대:', df['playtime_at_review_min'].max())
print('총 플레이타임 중앙값:', df['playtime_forever_min'].median(), '분 (약', round(forever_h), 'h ) / 최대:', df['playtime_forever_min'].max())
print('최근 2주 플레이 0인 비율:', round((df['playtime_2w_min'] == 0).mean() * 100, 1), '%')

games_owned 중앙값: 235.0 / 최대: 36496
num_reviews 최소: 1 / 중앙값: 14.0 / 최대: 25829
리뷰 시점 플레이타임 중앙값: 2258.0 분 (약 38 h ) / 최대: 503105
총 플레이타임 중앙값: 4628.0 분 (약 77 h ) / 최대: 1016252
최근 2주 플레이 0인 비율: 96.3 %


In [5]:
# 업적 파생 변수
print('리뷰 시점 2단계 비율:', round((df['stage_at_review'] == 2).mean() * 100, 1), '%')
print('리뷰 시점 업적 수 중앙값:', df['n_ach_at_review'].median())
print('총 업적 수 중앙값:', df['n_ach_total'].median(), '/ 최대:', df['n_ach_total'].max())
print('업적 82개(전부) 보유:', (df['n_ach_total'] == 82).sum(), '명')
print('최종 10단계 비율:', round((df['stage_final'] == 10).mean() * 100, 1), '%')

# 리뷰 시점 업적이 0개인 사람은 두 종류다: 끝까지 0개(업적 미추적) / 리뷰 전에는 0개였지만 뒤에 생김
zero_at_review = df['n_ach_at_review'] == 0
never = zero_at_review & (df['n_ach_total'] == 0)
later = zero_at_review & (df['n_ach_total'] > 0)
print('리뷰 시점 업적 0개:', zero_at_review.sum(), '= 미추적', never.sum(), '+ 리뷰 뒤에 생김', later.sum())

리뷰 시점 2단계 비율: 42.8 %
리뷰 시점 업적 수 중앙값: 10.0
총 업적 수 중앙값: 23.0 / 최대: 82
업적 82개(전부) 보유: 403 명
최종 10단계 비율: 39.8 %
리뷰 시점 업적 0개: 2526 = 미추적 999 + 리뷰 뒤에 생김 1527


In [6]:
# 메인 퀘스트 8·9단계 순서 확인: Entangled와 Unearthed를 둘 다 해금한 사람은 어느 쪽을 먼저 했나
ach = pd.read_csv('dataSets/starfield_achievements.csv', usecols=['steamid', 'name', 't'])
two = ach[ach['name'].isin(['Entangled', 'Unearthed'])]

# 한 사람(steamid)이 행, 업적 이름이 열, 값은 해금 시각이 되도록 표를 바꾼다
both = two.pivot_table(index='steamid', columns='name', values='t', aggfunc='min')
both = both.dropna()   # 둘 다 해금한 사람만

entangled_first = (both['Entangled'] < both['Unearthed']).sum()
unearthed_first = (both['Entangled'] > both['Unearthed']).sum()
print('둘 다 해금', len(both), '명 중 Entangled 먼저', entangled_first, '명 / Unearthed 먼저', unearthed_first, '명')

둘 다 해금 31119 명 중 Entangled 먼저 26982 명 / Unearthed 먼저 123 명


### 업적 0개 999명은 누구인가

총 플레이 5시간을 기준으로 둘로 나뉜다.
5시간 미만은 첫 업적을 따기 전에 그만둔 사람(조기 이탈), 5시간 이상인데 업적이 0개면 모드·콘솔 등으로 업적이 꺼진 계정으로 본다.

In [7]:
zero = df[df['n_ach_total'] == 0]
hours = zero['playtime_forever_min'] / 60

early = zero[hours < 5]
late = zero[hours >= 5]
print('업적 0개', len(zero), '명 = 5시간 미만', len(early), '명 (긍정률', round(early['voted_up'].mean() * 100), '%) + 5시간 이상', len(late), '명')
print('그중 100시간 이상:', (hours >= 100).sum(), '명')

# 참고: 업적이 추적되는 사람 중 5시간 넘게 했는데 리뷰 시점 업적이 0개인 비율 (아주 낮아야 정상)
normal = df[df['n_ach_total'] > 0]
played_5h = normal['playtime_at_review_min'] >= 300
no_ach = normal['n_ach_at_review'] == 0
print('정상 계정에서 5시간+ & 업적 0개 비율:', round((played_5h & no_ach).mean() * 100, 2), '%')

업적 0개 999 명 = 5시간 미만 458 명 (긍정률 23 %) + 5시간 이상 541 명
그중 100시간 이상: 153 명
정상 계정에서 5시간+ & 업적 0개 비율: 0.94 %


## 2. 제외 기준

| 기준 | 이유 |
|---|---|
| 총 업적 0개 | 업적이 추적되지 않아 진행 단계를 알 수 없다 |
| 보유 게임 0개 | Starfield를 갖고 있으니 0은 불가능 → 라이브러리 비공개 |
| 리뷰 시점 플레이 > 총 플레이 | 데이터 오류 |
| 리뷰 텍스트 없음 | 버튼만 누른 리뷰. 임베딩할 글이 없다 |

In [8]:
# 각 기준에 해당하는 행을 True/False로 표시한다
no_achievement = df['n_ach_total'] == 0
no_library = df['games_owned'] == 0
time_error = df['playtime_at_review_min'] > df['playtime_forever_min']
# review가 NaN이면 빈 문자열로 바꾼 뒤, 공백만 있는 것도 빈 것으로 본다
no_text = df['review'].fillna('').str.strip() == ''

print('업적 미추적     ', no_achievement.sum(), '건')
print('라이브러리 비공개', no_library.sum(), '건')
print('플레이타임 오류  ', time_error.sum(), '건')
print('리뷰 텍스트 없음 ', no_text.sum(), '건')

# 네 기준 중 하나라도 해당하면 제외
exclude = no_achievement | no_library | time_error | no_text
df = df[~exclude].reset_index(drop=True)

n_after = len(df)
print(n_before, '→', n_after, '행 (', round((n_before - n_after) / n_before * 100, 1), '% 제외 )')

업적 미추적      999 건
라이브러리 비공개 167 건
플레이타임 오류   2 건
리뷰 텍스트 없음  192 건
72271 → 70915 행 ( 1.9 % 제외 )


## 3. 저장 및 검증

저장한 파일을 다시 읽어서 행 수와 내용이 같은지 확인한다. CSV로 저장할 때 따옴표나 줄바꿈이 깨지면 여기서 걸린다.

In [9]:
out = df[['steamid', 'review', 'voted_up']]
out.to_csv('dataSets/reviews_for_embedding.csv', index=False, encoding='utf-8')

check = pd.read_csv('dataSets/reviews_for_embedding.csv')
print('행 수 같음:', len(check) == len(out))
print('steamid 같음:', (check['steamid'].values == out['steamid'].values).all())
print('리뷰 같음:', (check['review'].astype(str).values == out['review'].values).all())
print('voted_up 같음:', (check['voted_up'].values == out['voted_up'].values).all())
print('저장 완료:', len(out), '행')

행 수 같음: True
steamid 같음: True
리뷰 같음: True
voted_up 같음: True
저장 완료: 70915 행
